In [ ]:
COLS_ACCION = ["tiene_marcha", "tiene_corte", "tiene_paro", "tiene_acto",
               "tiene_escrache", "tiene_destrozos", "tiene_toma"]

import pandas as pd
import numpy as np

# Carga

# Crear intervencion_armonizada
def armonizar_intervencion(row):
    prov = row["provincia"]
    cod = row["tipo_intervencion_estatal"]
    
    if pd.isna(cod):
        return "Sin datos"
    
    cod = int(cod)
    
    if cod == 0:
        return "Ausencia"
    elif cod == 1:
        return "Monitoreo"
    elif cod == 2:
        return "Contención"
    elif cod == 3:
        # Chubut: Represión / Jujuy: Disuasión
        return "Represión" if prov == "Chubut" else "Disuasión"
    elif cod == 4:
        # Solo Jujuy: Represión
        return "Represión"
    else:
        return "Sin datos"

df["intervencion_armonizada"] = df.apply(armonizar_intervencion, axis=1)

# Verificar
print("=" * 60)
print("DISTRIBUCIÓN DE INTERVENCIÓN ARMONIZADA POR PROVINCIA")
print("=" * 60)
print(pd.crosstab(df["provincia"], df["intervencion_armonizada"]))

# Guardar
df.to_excel("datos_comparados.xlsx", index=False)
print("\n✓ Guardado en datos_comparados.xlsx")

print("TIPO DE ACCIÓN × INTERVENCIÓN ESTATAL\n")

for prov in ["Chubut", "Jujuy"]:
    sub = df[df["provincia"] == prov].copy()
    
    # DEBUG: verificar que el filtro funciona
    print(f"=== {prov} (n total = {len(sub)}) ===")
    
    filas = []
    for accion in COLS_ACCION:
        sub_acc = sub[sub[accion] == 1]
        n = len(sub_acc)
        if n >= 3:
            conteo = sub_acc["intervencion_armonizada"].value_counts().to_dict()
            fila = {"acción": accion.replace("tiene_", ""), "n": n}
            fila.update({k: conteo.get(k, 0) for k in 
                         ["Ausencia", "Monitoreo", "Contención", "Disuasión", "Represión", "Sin datos"]})
            filas.append(fila)
    
    tabla = pd.DataFrame(filas).set_index("acción")
    print(tabla.to_string())
    print()


print("TIPO DE ACCIÓN × TIPO DE ACTORES\n")

for prov in ["Chubut", "Jujuy"]:
    sub = df[df["provincia"] == prov].copy()
    
    # DEBUG: verificar que el filtro funciona
    print(f"=== {prov} (n total = {len(sub)}) ===")
    
    filas = []
    for accion in COLS_ACCION:
        sub_acc = sub[sub[accion] == 1]
        n = len(sub_acc)
        if n >= 3:
            conteo = sub_acc["intervencion_armonizada"].value_counts().to_dict()
            fila = {"acción": accion.replace("tiene_", ""), "n": n}
            fila.update({k: conteo.get(k, 0) for k in 
                         ["Ausencia", "Monitoreo", "Contención", "Disuasión", "Represión", "Sin datos"]})
            filas.append(fila)
    
    tabla = pd.DataFrame(filas).set_index("acción")
    print(tabla.to_string())
    print()


def tabla_accion_intervencio(prov):
    """Tipo de accion X intervencion estatal para una provincia"""
    sub = df[df["provincia"] == prov]
    filas = []
    for accion in COLS_ACCION:
        sub_acc = sub[sub[accion] == 1]
        n = len(sub_acc)
        if n >= 3:
            conteo = sub_acc['intervencion_armonizada'].value_counts().to_dict()
            fila = {"accion": accion.replace("tiene_",""), "n": n}
            fila.update({k: conteo.get(k,0) for k in CAST_INTERV})
            filas.pappend(fila)
        return pd.DataFrame(filas).set_index("accion")


def tabla_actor_accion(prov):
    sub = df[df["provincia"] == prov]
    filas = []
    for actor in COLS_ACTORES:
        sub_act = sub[sub[actor] == 1]
        if len(sub_act) >= 3:
            fila = {a.replace("tiene_", ""): sub_act[a].sum() for a in COLS_ACCION}
            fila["n"] = len(sub_act)
            filas.append({"actor": actor, **fila})
    return pd.DataFrame(filas).set_index("actor")

print("=" * 70)
print("CROSSTAB 4: ACTOR × TIPO DE ACCIÓN")
print("=" * 70)

for prov in ["Chubut", "Jujuy"]:
    print(f"\n=== {prov} ===")
    print(tabla_actor_accion(prov).to_string())

def tabla_accion_intervencion(prov):
    """Tipo de acción × Intervención estatal para una provincia."""
    sub = df[df["provincia"] == prov]
    filas = []
    for accion in COLS_ACCION:
        sub_acc = sub[sub[accion] == 1]
        n = len(sub_acc)
        if n >= 3:
            conteo = sub_acc["intervencion_armonizada"].value_counts().to_dict()
            fila = {"acción": accion.replace("tiene_", ""), "n": n}
            fila.update({k: conteo.get(k, 0) for k in CATS_INTERV})
            filas.append(fila)
    return pd.DataFrame(filas).set_index("acción")


def tabla_actor_represion(prov):
    """Actor × % represión para una provincia."""
    sub = df[df["provincia"] == prov]
    filas = []
    for actor in COLS_ACTORES:
        sub_act = sub[sub[actor] == 1]
        n = len(sub_act)
        if n >= 3:
            n_rep = sub_act["es_represion"].sum()
            filas.append({
                "actor": actor, "n": n, "n_rep": n_rep,
                "%_rep": round(n_rep/n*100, 1)
            })
    return pd.DataFrame(filas).set_index("actor")


def tabla_actor_accion(prov):
    """Actor × Tipo de acción (booleanos) para una provincia."""
    sub = df[df["provincia"] == prov]
    filas = []
    for actor in COLS_ACTORES:
        sub_act = sub[sub[actor] == 1]
        if len(sub_act) >= 3:
            fila = {a.replace("tiene_", ""): sub_act[a].sum() for a in COLS_ACCION}
            fila["n"] = len(sub_act)
            filas.append({"actor": actor, **fila})
    return pd.DataFrame(filas).set_index("actor")


def tabla_demanda_accion(prov):
    """Demanda × Tipo de acción (booleanos) para una provincia."""
    sub = df[df["provincia"] == prov]
    filas = []
    for dem in COLS_DEMANDAS:
        sub_dem = sub[sub[dem] == 1]
        if len(sub_dem) >= 3:
            fila = {a.replace("tiene_", ""): sub_dem[a].sum() for a in COLS_ACCION}
            fila["n"] = len(sub_dem)
            filas.append({"demanda": dem.replace("demanda_", ""), **fila})
    return pd.DataFrame(filas).set_index("demanda")